# 06 - Termómetro de la economía de los hogares (EPH)

Índice compuesto trimestral que resume **cómo le está yendo a los hogares** según la EPH.
Funciona como un termómetro de **malestar**: **0 = mejor trimestre de la serie, 100 = peor**
(más "fiebre"). Se construye con los parquets por trimestre compilados por el notebook
**00_preparacion_bases**.

**Tres dimensiones (etapa 1: sin variables monetarias, no requiere deflactar):**

| Dimensión | Indicador | Variables EPH | Base |
|---|---|---|---|
| **A. Cantidad de empleo** | Desocupación | `ESTADO` | % PEA |
| | Tasa de empleo (invertida) | `ESTADO` | % población |
| | Desocupación de larga duración (>1 año buscando) | `ESTADO==2` & `PP10A==5` | % PEA |
| | Desocupados por despido/cierre o renuncia obligada/pactada (ex asalariados) | `ESTADO==2` & `PP11O∈{1,7}` | % PEA |
| **B. Calidad del empleo** | Subocupación horaria | `INTENSI==1` | % PEA |
| | Ocupados demandantes de otro empleo | `ESTADO==1` & `PP03J==1` | % PEA |
| | Asalariados sin descuento jubilatorio | `CAT_OCUP==3` & `PP07H==2` | % asalariados |
| | Tasa de empleo asalariado registrado (invertida) | `CAT_OCUP==3` & `PP07H==1` | % población |
| **C. Estrés de los hogares** (últimos 3 meses) | Gastaron ahorros | `V13==1` | % hogares |
| | Pidieron préstamos a familiares/amigos | `V14==1` | % hogares |
| | Vendieron pertenencias | `V17==1` | % hogares |
| | Recibieron mercadería/alimentos del gobierno, iglesias, escuelas | `V6==1` | % hogares |
| | Recibieron mercadería/alimentos de familiares/vecinos | `V7==1` | % hogares |

**Metodología:**
1. Cada indicador se calcula por trimestre, ponderado con `PONDERA` (hogares: un registro
   por hogar, el jefe/a `CH03==1`).
2. Se orienta para que "más alto = peor" (las tasas de empleo se invierten) y se normaliza
   como **percentil dentro de su propia historia** 2017-hoy (0 = mejor valor observado,
   100 = peor). Percentiles en lugar de min-max para que un outlier (2020T2) no aplaste
   el resto de la serie. Antes de rankear se redondea a 1 decimal (diferencias < 0,1 pp
   son ruido muestral y empatan).
3. Dimensión = promedio de sus indicadores; **Termómetro = promedio simple de las 3
   dimensiones** (cada dimensión pesa 1/3).
4. Franjas: **Templado** (≤ 33), **Tibio** (≤ 66), **Fiebre** (> 66).

**Cómo leerlo:** el valor es *relativo a la historia disponible* (2017-hoy): 60 significa
"peor que el 60% de los trimestres observados", no un nivel absoluto. Al sumar trimestres
los valores pasados pueden moverse levemente. Hay estacionalidad (el T1 suele ser peor), por
eso conviene mirar la **variación interanual** y la **media móvil de 4 trimestres**.

**Quedan fuera del índice (se muestran como complementarios):**
- Préstamos bancarios (`V15`) y compras en cuotas/fiado (`V16`): suben tanto por estrés como
  por mayor acceso al crédito (señal ambigua).
- Informalidad `EMPLEO`: solo existe desde 2023T4 (en el índice se usa `PP07H`, serie larga).
- Pluriempleo (`PP03C==2`): evaluado y descartado (Anexo A): es procíclico, cae en la
  pandemia y en cada T1; se comporta como un indicador de empleo, no de estrés.
- Desalentados (`PP02E==3`) y niños <10 que aportan dinero (`V19_A/B`): niveles de 0,0-0,4%,
  donde la diferencia entre trimestres es ruido muestral; además los desalentados *bajan*
  en 2020 (en cuarentena no se buscaba trabajo por otras razones).

**Efecto composición (por qué hay una tasa sobre población en la dimensión B):** en
2020T2 se perdieron sobre todo empleos informales y de pocas horas, así que los indicadores
calculados *sobre los ocupados* (informalidad, subocupación, ocupados que buscan otro empleo)
**mejoraron** artificialmente. La tasa de empleo asalariado registrado sobre la población no
tiene ese sesgo y compensa la dimensión.

**Alcance y advertencias:** no usa ingresos ni pobreza (no depende de deflactores; los
ingresos están en el notebook 03). La EPH cubre 31 aglomerados urbanos y se publica ~3 meses
después del trimestre; 2020T2-T3 se relevó telefónicamente (pandemia).

**Contenido:** 1-4 preparación (setup, carga, indicadores, normalización) · 5-6 el
termómetro hoy (gráfico y lectura automática) · 7-9 evolución (índice, dimensiones, mapa de
calor) · 10 tabla del último trimestre · 11 promedios anuales · 12 controles de calidad ·
13 exportar · Anexo A diagnóstico de candidatas · Anexo B historial de versiones.

*Versión del índice: v3 (2026-09).*

## 1. Setup (Colab)

In [ ]:
import sys, os, shutil, glob

REPO_URL = "https://github.com/santiagoriverti/analisis_EPH.git"
REPO_DIR = "/content/analisis_EPH"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount('/content/drive')

# Copia única Drive -> disco local (leer muchos parquets por FUSE desconecta el mount).
DRIVE_PROCESSED = "/content/drive/MyDrive/carga_EPH/processed"
PROCESSED_DIR = "/content/processed_local"
os.makedirs(PROCESSED_DIR, exist_ok=True)

for src in glob.glob(os.path.join(DRIVE_PROCESSED, "eph_T*.parquet")):
    dst = os.path.join(PROCESSED_DIR, os.path.basename(src))
    if not os.path.exists(dst):
        shutil.copy(src, dst)

n = len(glob.glob(os.path.join(PROCESSED_DIR, "eph_T*.parquet")))
print(f"Parquets locales listos en {PROCESSED_DIR}: {n} trimestres")

# Carpeta de Drive donde se guarda la tabla del termómetro (CSV)
RESULTADOS_DIR = "/content/drive/MyDrive/carga_EPH/resultados"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from src.data_loader import load_panel, list_available_quarters, _period_tag

plt.rcParams.update({
    "figure.figsize": (11, 5.5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e3df",
    "grid.linewidth": 0.8,
    "axes.edgecolor": "#8a8984",
    "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e",
    "ytick.color": "#52514e",
})

# Paleta (validada para daltonismo): una por dimensión + tinta para el índice.
COLOR_DIM = {"A. Cantidad de empleo": "#2a78d6",
             "B. Calidad del empleo": "#eb6834",
             "C. Estrés de los hogares": "#1baf7a"}
TINTA = "#0b0b0b"
# Franjas del termómetro (colores de estado, siempre acompañados de etiqueta)
FRANJAS = [(0, 33, "#0ca30c", "Templado"), (33, 66, "#fab219", "Tibio"),
           (66, 100, "#d03b3b", "Fiebre")]

quarters = list_available_quarters()
ULTIMO = quarters[-1]
print("Trimestres disponibles:", len(quarters), "| último:", _period_tag(*ULTIMO))

## 2. Carga de datos

Personas (mercado laboral) y hogares (estrategias del hogar, un registro por hogar).
Todas las columnas se fuerzan a numéricas (ver gotcha de dtypes en `docs/TECNICO.md`).

In [ ]:
PERS_COLS = ["ESTADO", "INTENSI", "PP10A", "PP02E", "PP03J", "PP07H", "CAT_OCUP", "EMPLEO",
             "PP11O", "PP03C", "PONDERA"]
HOG_COLS = ["CH03", "V6", "V7", "V13", "V14", "V15", "V16", "V17", "V19_A", "V19_B", "PONDERA"]

def cargar(cols):
    df = load_panel(columns=cols, out_dir=PROCESSED_DIR)
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        else:
            df[c] = np.nan  # columna inexistente en todos los trimestres
    return df

pers = cargar(PERS_COLS)
hog = cargar(HOG_COLS)
hog = hog[hog["CH03"] == 1].drop(columns="CH03")
print(f"Personas: {len(pers):,} filas | Hogares: {len(hog):,} filas")

## 3. Indicadores por trimestre

Todos en %. "Más alto = peor" salvo la tasa de empleo (se invierte al normalizar).

In [ ]:
def pct(w, num, den):
    """% ponderado: suma de w donde num&den, sobre suma de w donde den."""
    d = w[den].sum()
    return w[num & den].sum() / d * 100 if d > 0 else np.nan

def ind_personas(g):
    w, e = g["PONDERA"], g["ESTADO"]
    todos = pd.Series(True, index=g.index)
    ocup, desoc = e == 1, e == 2
    pea = ocup | desoc
    asal = ocup & (g["CAT_OCUP"] == 3)
    d = w[pea].sum()
    return pd.Series({
        "Desocupación": pct(w, desoc, pea),
        "Tasa de empleo": pct(w, ocup, todos),
        "Desocupación >1 año": w[desoc & (g["PP10A"] == 5)].sum() / d * 100,
        "Desalentados": w[(e == 3) & (g["PP02E"] == 3)].sum() / d * 100,
        "Subocupación": w[g["INTENSI"] == 1].sum() / d * 100,
        "Ocupados que buscan otro empleo": w[ocup & (g["PP03J"] == 1)].sum() / d * 100,
        "Asalariados sin descuento jubilatorio": pct(w, g["PP07H"] == 2, asal),
        "Tasa de empleo asalariado registrado": pct(w, asal & (g["PP07H"] == 1), todos),
        "Desocupados por despido (incl. renuncia forzada)": w[desoc & g["PP11O"].isin([1, 7])].sum() / d * 100,  # 1 despido/cierre, 7 renuncia obligada/pactada
        # complementario (evaluado y descartado para el índice, ver Anexo A)
        "Pluriempleo": pct(w, g["PP03C"] == 2, ocup),
        # complementario (solo desde 2023T4)
        "Informalidad (EMPLEO)": pct(w, g["EMPLEO"] == 2, ocup) if g["EMPLEO"].notna().any() else np.nan,
    })

def ind_hogares(g):
    w = g["PONDERA"]
    todos = pd.Series(True, index=g.index)
    return pd.Series({
        "Gastaron ahorros": pct(w, g["V13"] == 1, todos),
        "Préstamos de familiares/amigos": pct(w, g["V14"] == 1, todos),
        "Vendieron pertenencias": pct(w, g["V17"] == 1, todos),
        "Alimentos de gobierno/instituciones": pct(w, g["V6"] == 1, todos),
        "Alimentos de familiares/vecinos": pct(w, g["V7"] == 1, todos),
        "Niños <10 aportan dinero": pct(w, (g["V19_A"] == 1) | (g["V19_B"] == 1), todos),
        # complementarios (señal ambigua: estrés o más acceso al crédito)
        "Préstamos de bancos/financieras": pct(w, g["V15"] == 1, todos),
        "Compras en cuotas/fiado": pct(w, g["V16"] == 1, todos),
    })

ind = pd.concat([
    pers.groupby(["ANIO", "TRIMESTRE"]).apply(ind_personas, include_groups=False),
    hog.groupby(["ANIO", "TRIMESTRE"]).apply(ind_hogares, include_groups=False),
], axis=1).sort_index()
ind.index = [f"{y}T{p}" for y, p in ind.index]
ind.round(1)

## 4. Normalización y armado del termómetro

Cada indicador se convierte en su **percentil histórico orientado** (0 = mejor trimestre
de la serie, 100 = peor).

In [ ]:
DIMENSIONES = {
    "A. Cantidad de empleo": {"Desocupación": +1, "Tasa de empleo": -1,
                              "Desocupación >1 año": +1, "Desocupados por despido (incl. renuncia forzada)": +1},
    "B. Calidad del empleo": {"Subocupación": +1, "Ocupados que buscan otro empleo": +1,
                              "Asalariados sin descuento jubilatorio": +1,
                              "Tasa de empleo asalariado registrado": -1},
    "C. Estrés de los hogares": {"Gastaron ahorros": +1, "Préstamos de familiares/amigos": +1,
                                 "Vendieron pertenencias": +1,
                                 "Alimentos de gobierno/instituciones": +1,
                                 "Alimentos de familiares/vecinos": +1},
}
COMPLEMENTARIOS = ["Informalidad (EMPLEO)", "Préstamos de bancos/financieras", "Compras en cuotas/fiado",
                   "Desalentados", "Niños <10 aportan dinero",
                   "Pluriempleo"]
SIGNO = {c: s for cols in DIMENSIONES.values() for c, s in cols.items()}  # indicador -> signo

def percentil_orientado(s, signo):
    """0 = mejor valor de la serie, 100 = peor.

    Se redondea a 1 decimal antes de rankear: diferencias menores a 0,1 pp son ruido
    muestral, y así los empates quedan en el percentil promedio en vez de en los extremos
    (importa en indicadores de nivel muy bajo, como desalentados o niños que aportan).
    """
    r = (signo * s.round(1)).rank(method="average")
    n = r.notna().sum()
    return (r - 1) / (n - 1) * 100 if n > 1 else r * np.nan

def armar_termometro(dimensiones):
    """Devuelve (percentiles por indicador, dimensiones, termómetro) para un set de indicadores."""
    norm = pd.DataFrame({ind_name: percentil_orientado(ind[ind_name], signo)
                         for dim in dimensiones.values() for ind_name, signo in dim.items()})
    dims = pd.DataFrame({dim: norm[list(cols)].mean(axis=1) for dim, cols in dimensiones.items()})
    return norm, dims, dims.mean(axis=1)

norm, dims, t = armar_termometro(DIMENSIONES)
termo = dims.copy()
termo["Termómetro"] = t
termo["Media móvil 4T"] = termo["Termómetro"].rolling(4).mean()
termo["Var. interanual (pts)"] = termo["Termómetro"].diff(4)

def franja(v):
    for lo, hi, _, nombre in FRANJAS:
        if v <= hi:
            return nombre
    return FRANJAS[-1][3]

termo["Estado"] = termo["Termómetro"].map(franja)
termo.round(1)

## 5. El termómetro hoy

Valor del último trimestre, comparado con el mismo trimestre del año anterior
(comparación interanual: neutraliza la estacionalidad).

In [ ]:
ult = termo.index[-1]
hace_un_anio = termo.index[-5] if len(termo) >= 5 else None
v = termo.loc[ult, "Termómetro"]

fig, (ax_t, ax_d) = plt.subplots(1, 2, figsize=(11, 5.5), gridspec_kw={"width_ratios": [1, 3]})

# Termómetro vertical
for lo, hi, col, nombre in FRANJAS:
    ax_t.axhspan(lo, hi, xmin=0.3, xmax=0.7, color=col, alpha=0.18, lw=0)
    ax_t.text(0.75, (lo + hi) / 2, nombre, transform=ax_t.get_yaxis_transform(),
              va="center", color="#52514e", fontsize=10)
ax_t.bar(0.5, v, width=0.28, color=TINTA, zorder=3)
ax_t.scatter([0.5], [0], s=900, color=TINTA, zorder=3)
ax_t.text(0.5, v + 3, f"{v:.0f}", ha="center", va="bottom", fontsize=22, fontweight="bold", color=TINTA)
ax_t.set_xlim(0, 1.3); ax_t.set_ylim(-6, 108)
ax_t.set_xticks([]); ax_t.set_yticks([0, 33, 66, 100]); ax_t.grid(False)
ax_t.spines["bottom"].set_visible(False)
ax_t.set_title(f"Termómetro {ult}\n({franja(v)})", color=TINTA)

# Dimensiones: último trimestre vs. mismo trimestre del año anterior
nombres = list(DIMENSIONES)
y = np.arange(len(nombres))
if hace_un_anio is not None:
    ax_d.scatter(dims.loc[hace_un_anio, nombres], y, s=90, facecolors="none",
                 edgecolors="#8a8984", linewidths=2, label=hace_un_anio, zorder=3)
ax_d.scatter(dims.loc[ult, nombres], y, s=110, c=[COLOR_DIM[n] for n in nombres],
             edgecolors="#fcfcfb", linewidths=2, label=ult, zorder=4)
for i, n in enumerate(nombres):
    ax_d.text(dims.loc[ult, n], i - 0.18, f"{dims.loc[ult, n]:.0f}", ha="center", va="bottom", color=TINTA)
ax_d.set_yticks(y, nombres); ax_d.set_ylim(len(nombres) - 0.5, -0.7)
ax_d.set_xlim(0, 100); ax_d.set_xlabel("Percentil histórico (0 = mejor, 100 = peor)")
ax_d.set_title("Dimensiones")
ax_d.legend(loc="lower right", frameon=False)
plt.tight_layout()
plt.show()

if hace_un_anio is not None:
    delta = termo.loc[ult, "Var. interanual (pts)"]
    print(f"{ult}: {v:.1f} ({franja(v)}) | {hace_un_anio}: {termo.loc[hace_un_anio, 'Termómetro']:.1f} "
          f"| variación interanual: {delta:+.1f} pts")

## 6. Lectura del último trimestre

Resumen en texto generado a partir de los datos (se actualiza solo con cada trimestre nuevo
y se exporta en la sección 13). Incluye la posición del trimestre en la serie, qué
dimensión explica la variación interanual (cada dimensión aporta 1/3 de su cambio) y qué
indicadores están en niveles extremos o se movieron más.

In [ ]:
from IPython.display import Markdown, display

def f1(x, signo=False):
    """Número con 1 decimal y coma decimal (signo explícito opcional)."""
    return (f"{x:+.1f}" if signo else f"{x:.1f}").replace(".", ",").replace("-", "−")

def posicion(serie, q, fem=False):
    """Ranking de q en la serie y desde cuándo no había un valor tan alto (o tan bajo).

    Devuelve (texto del ranking, texto "desde cuándo" o None si no hay trimestres previos).
    """
    serie = serie.dropna()
    v, n = serie[q], len(serie)
    alto = int(serie.rank(ascending=False, method="min")[q])
    bajo = int(serie.rank(ascending=True, method="min")[q])
    palabra = "alto" if alto <= bajo else "bajo"
    art = ("la más " + palabra[:-1] + "a") if fem else ("el valor más " + palabra)
    ranking = f"{min(alto, bajo)}.º trimestre más {palabra} de {n}"
    previos = serie.loc[:q].iloc[:-1]
    if previos.empty:
        return ranking, None
    iguales = previos[previos >= v] if palabra == "alto" else previos[previos <= v]
    if iguales.empty:
        return ranking, f"{art} de la serie" + ("" if q == serie.index[-1] else " hasta entonces")
    return ranking, f"{art} desde {iguales.index[-1]}"

def resumen_trimestre(q):
    t_ = termo["Termómetro"]
    ma = termo["Media móvil 4T"]
    i = termo.index.get_loc(q)
    q_ia = termo.index[i - 4] if i >= 4 else None
    q_ant = termo.index[i - 1] if i >= 1 else None
    L = [f"### Termómetro {q}: {f1(t_[q])} ({franja(t_[q])})", ""]

    ranking, desde = posicion(t_, q)
    L.append(f"- **Posición:** {ranking}" + (f" ({desde})." if desde else "."))
    if q_ia is not None:
        contrib = ((dims.loc[q] - dims.loc[q_ia]) / len(DIMENSIONES)).sort_values(key=abs, ascending=False)
        partes = ", ".join(f"{n.split('. ')[1].lower()} {f1(c, True)}" for n, c in contrib.items())
        L.append(f"- **Interanual:** {f1(t_[q_ia])} en {q_ia} → {f1(t_[q])} "
                 f"(**{f1(t_[q] - t_[q_ia], True)} pts**). Aporte por dimensión: {partes}.")
    if q_ant is not None:
        L.append(f"- **Trimestral:** {f1(t_[q] - t_[q_ant], True)} pts frente a {q_ant} "
                 f"(afectado por estacionalidad).")
    if pd.notna(ma[q]):
        _, desde = posicion(ma, q, fem=True)
        L.append(f"- **Media móvil 4T:** {f1(ma[q])}" + (f" ({desde})." if desde else "."))
    L.append("- **Dimensiones:** " + " · ".join(
        f"{n} {f1(dims.loc[q, n])} ({franja(dims.loc[q, n])})" for n in DIMENSIONES) + ".")

    # Indicadores en el peor / mejor valor de la serie (con el redondeo a 1 decimal del índice)
    orient = pd.DataFrame({c: s * ind[c].round(1) for c, s in SIGNO.items()})
    peor = [c for c in SIGNO if orient.loc[q, c] == orient[c].max()]
    mejor = [c for c in SIGNO if orient.loc[q, c] == orient[c].min()]
    if peor:
        L.append("- **En su peor nivel de la serie:** " +
                 "; ".join(f"{c} ({f1(ind.loc[q, c])}%)" for c in peor) + ".")
    if mejor:
        L.append("- **En su mejor nivel de la serie:** " +
                 "; ".join(f"{c} ({f1(ind.loc[q, c])}%)" for c in mejor) + ".")
    altos = norm.loc[q].loc[lambda s: (s >= 80)].sort_values(ascending=False).drop(peor, errors="ignore")
    if len(altos):
        L.append("- **Otros indicadores altos (percentil ≥ 80):** " +
                 "; ".join(f"{c} {f1(ind.loc[q, c])}% (p{norm.loc[q, c]:.0f})" for c in altos.index) + ".")

    if q_ia is not None:
        mov = (norm.loc[q] - norm.loc[q_ia]).sort_values()
        def txt(cs):
            return "; ".join(f"{c} {f1(ind.loc[q, c])}% ({f1(ind.loc[q, c] - ind.loc[q_ia, c], True)} pp, "
                             f"p{norm.loc[q_ia, c]:.0f}→p{norm.loc[q, c]:.0f})" for c in cs)
        subas = mov[mov >= 20].index[::-1][:3]
        bajas = mov[mov <= -20].index[:3]
        if len(subas):
            L.append(f"- **Mayores deterioros interanuales:** {txt(subas)}.")
        if len(bajas):
            L.append(f"- **Mayores mejoras interanuales:** {txt(bajas)}.")
        comp = [c for c in COMPLEMENTARIOS
                if pd.notna(ind.loc[q, c]) and pd.notna(ind.loc[q_ia, c])
                and abs(ind.loc[q, c] - ind.loc[q_ia, c]) >= 1]
        if comp:
            L.append("- **Complementarios que se movieron ≥ 1 pp i.a.:** " + "; ".join(
                f"{c} {f1(ind.loc[q, c])}% ({f1(ind.loc[q, c] - ind.loc[q_ia, c], True)} pp)" for c in comp) + ".")
    return "\n".join(L)

resumen = resumen_trimestre(ult)
display(Markdown(resumen))

## 7. Evolución del termómetro

Índice trimestral, media móvil de 4 trimestres (suaviza la estacionalidad) y franjas.

In [ ]:
x = np.arange(len(termo))
fig, ax = plt.subplots(figsize=(12, 5.5))
for lo, hi, col, nombre in FRANJAS:
    ax.axhspan(lo, hi, color=col, alpha=0.08, lw=0)
    ax.text(len(termo) - 0.5, (lo + hi) / 2, nombre, va="center", ha="left", color="#52514e")
ax.plot(x, termo["Termómetro"], color="#8a8984", lw=1.5, marker="o", ms=4, label="Trimestral")
ax.plot(x, termo["Media móvil 4T"], color=TINTA, lw=2.5, label="Media móvil 4T")
i_max = int(np.nanargmax(termo["Termómetro"]))
i_min = int(np.nanargmin(termo["Termómetro"]))
for i in {i_max, i_min, len(termo) - 1}:
    ax.annotate(f"{termo.index[i]}: {termo['Termómetro'].iloc[i]:.0f}", (x[i], termo["Termómetro"].iloc[i]),
                xytext=(0, 10), textcoords="offset points", ha="center", color=TINTA, fontsize=9)
ax.set_xticks(x, termo.index, rotation=90)
ax.set_xlim(-0.5, len(termo) + 2); ax.set_ylim(0, 100)
ax.set_ylabel("Termómetro (0 = mejor, 100 = peor)")
ax.set_title("Termómetro de la economía de los hogares - EPH")
ax.legend(loc="upper left", frameon=False)
plt.tight_layout()
plt.show()

## 8. Evolución por dimensión

Una línea por dimensión (percentil histórico promedio de sus indicadores, media móvil 4T).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.5))
suav = dims.rolling(4).mean()
for n, col in COLOR_DIM.items():
    ax.plot(x, suav[n], color=col, lw=2, label=n)
# Etiquetas directas al final de cada línea, separadas para que no se pisen
fin = suav.iloc[-1].sort_values()
pos, prev = {}, -np.inf
for n, val in fin.items():
    pos[n] = max(val, prev + 5)
    prev = pos[n]
for n in fin.index:
    ax.text(x[-1] + 0.4, pos[n], f"{n.split('. ')[1]} ({fin[n]:.0f})", color="#52514e", va="center", fontsize=9)
ax.set_xticks(x, termo.index, rotation=90)
ax.set_xlim(-0.5, len(termo) + 5); ax.set_ylim(0, 100)
ax.set_ylabel("Percentil histórico (media móvil 4T)")
ax.set_title("Dimensiones del termómetro (0 = mejor, 100 = peor)")
ax.legend(loc="upper left", frameon=False)
plt.tight_layout()
plt.show()

## 9. Mapa de calor de los indicadores

Cada celda es el percentil histórico del indicador en ese trimestre (más oscuro = peor).
Permite ver qué indicadores "empujan" el termómetro en cada momento.

In [ ]:
orden = [c for cols in DIMENSIONES.values() for c in cols]
fig, ax = plt.subplots(figsize=(13, 6.5))
im = ax.imshow(norm[orden].T.values, aspect="auto", cmap="Reds", vmin=0, vmax=100)
ax.set_yticks(range(len(orden)), orden)
ax.set_xticks(range(len(norm)), norm.index, rotation=90)
ax.grid(False)
# Separadores entre dimensiones
acum = 0
for cols in list(DIMENSIONES.values())[:-1]:
    acum += len(cols)
    ax.axhline(acum - 0.5, color="#fcfcfb", lw=3)
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01)
cb.set_label("Percentil (100 = peor)")
ax.set_title("Indicadores normalizados por trimestre")
plt.tight_layout()
plt.show()

## 10. Tabla del último trimestre

Valor de cada indicador (en %), el mismo trimestre del año anterior, la diferencia
(puntos porcentuales) y su percentil histórico. Incluye los indicadores complementarios
(fuera del índice, sin percentil).

In [ ]:
filas = []
for dim, cols in DIMENSIONES.items():
    for c, signo in cols.items():
        filas.append((dim, c, signo))
for c in COMPLEMENTARIOS:
    filas.append(("Complementario (fuera del índice)", c, +1))

tabla = pd.DataFrame([{
    "Dimensión": dim,
    "Indicador": c,
    ult: ind.loc[ult, c],
    **({hace_un_anio: ind.loc[hace_un_anio, c],
        "Dif. i.a. (pp)": ind.loc[ult, c] - ind.loc[hace_un_anio, c]} if hace_un_anio else {}),
    "Percentil histórico": norm.loc[ult, c] if c in norm else np.nan,
    "Sentido": "↑ peor" if signo > 0 else "↓ peor",
} for dim, c, signo in filas]).set_index(["Dimensión", "Indicador"])
tabla.round(1)

## 11. Promedios anuales

Promedio de los trimestres de cada año (dimensiones y termómetro). El año en curso suele
estar incompleto (columna `Trimestres`): no es comparable con un año completo porque el
T1 tiende a ser peor.

In [ ]:
anio = termo.index.str[:4]
anual = termo[[*DIMENSIONES, "Termómetro"]].groupby(anio).mean()
anual.insert(0, "Trimestres", termo.groupby(anio).size())
anual["Estado"] = anual["Termómetro"].map(franja)
anual.index.name = "Año"

col_franja = {nombre: col for _, _, col, nombre in FRANJAS}
fig, ax = plt.subplots(figsize=(11, 4.8))
xa = np.arange(len(anual))
for i, (a, r) in enumerate(anual.iterrows()):
    incompleto = r["Trimestres"] < 4
    ax.bar(i, r["Termómetro"], color=col_franja[r["Estado"]], alpha=0.35 if incompleto else 0.85,
           hatch="//" if incompleto else None, edgecolor="#8a8984" if incompleto else "none", width=0.7)
    ax.text(i, r["Termómetro"] + 1.5, f"{r['Termómetro']:.0f}", ha="center", va="bottom", color=TINTA)
ax.set_xticks(xa, [a if n == 4 else f"{a}\n({n}T)" for a, n in zip(anual.index, anual["Trimestres"])])
ax.set_ylim(0, 100); ax.set_ylabel("Termómetro (promedio anual)")
ax.set_title("Termómetro por año")
ax.legend(handles=[mpatches.Patch(color=col, alpha=0.85, label=nombre) for _, _, col, nombre in FRANJAS],
          loc="upper left", frameon=False, ncol=3)
plt.tight_layout()
plt.show()

anual.round(1)

## 12. Controles de calidad

Chequeos automáticos para correr después de sumar un trimestre (✓ = OK, ⚠ = revisar):

1. **Cobertura:** todos los indicadores del índice tienen dato en todos los trimestres (un
   faltante en el último trimestre suele indicar una columna que cambió de nombre en la base).
2. **Tamaño de muestra:** el último trimestre tiene una cantidad de hogares normal (≥ 80% de
   la mediana), para detectar cargas incompletas.
3. **Chequeo de sentido:** el índice tiene que reconocer los episodios conocidos: máximo en
   la crisis 2019-2021, pandemia (2020T2-2021T1) en Fiebre en promedio, valle 2023T3-T4 en
   Templado.
4. **Códigos estables de `PP11O`** (indicador de despidos): distribución entre desocupados
   antes y después del cambio de esquema de 4T2023 (referencia: código 1 = 12,7% → 14,2%,
   código 7 = 1,5% → 2,2%).

In [ ]:
PANDEMIA = ["2020T2", "2020T3", "2020T4", "2021T1"]
VALLE = ["2023T3", "2023T4"]

def chequeo(ok, texto):
    print(("✓ " if ok else "⚠ ") + texto)

# 1. Cobertura
faltan = ind[list(SIGNO)].isna()
chequeo(not faltan.values.any(),
        "Cobertura completa de los indicadores del índice" if not faltan.values.any() else
        "Faltan datos: " + "; ".join(f"{c} en {', '.join(faltan.index[faltan[c]])}"
                                     for c in faltan.columns if faltan[c].any()))

# 2. Tamaño de muestra
n_hog = hog.groupby(["ANIO", "TRIMESTRE"]).size()
n_hog.index = [f"{a}T{p}" for a, p in n_hog.index]
chequeo(n_hog.iloc[-1] >= 0.8 * n_hog.median(),
        f"Hogares en {n_hog.index[-1]}: {n_hog.iloc[-1]:,} (mediana de la serie {n_hog.median():,.0f})".replace(",", "."))

# 3. Chequeo de sentido
t_ = termo["Termómetro"]
q_max = t_.idxmax()
chequeo("2019" <= q_max[:4] <= "2021", f"Máximo de la serie: {q_max} ({f1(t_.max())}), esperado en 2019-2021")
if set(PANDEMIA) <= set(t_.index):
    m = t_.loc[PANDEMIA].mean()
    chequeo(m > 66, f"Pandemia 2020T2-2021T1: promedio {f1(m)} (esperado > 66, Fiebre)")
if set(VALLE) <= set(t_.index):
    m = t_.loc[VALLE].mean()
    chequeo(m <= 33, f"Valle 2023T3-T4: promedio {f1(m)} (esperado ≤ 33, Templado)")

# 4. Códigos de PP11O entre desocupados, por esquema
pers["esquema"] = np.where(pers["ANIO"] * 10 + pers["TRIMESTRE"] >= 20234, "desde 2023T4", "hasta 2023T3")
sub = pers[pers["ESTADO"] == 2]
dist = (sub.groupby(["esquema", "PP11O"])["PONDERA"].sum()
           .groupby(level=0).transform(lambda s: s / s.sum() * 100).unstack(0))
print("\nPP11O entre desocupados (% ponderado; 1 = despido/cierre, 7 = renuncia obligada/pactada):")
print(dist.round(1).to_string())

## 13. Exportar

Guarda en Drive (`carga_EPH/resultados/`), con separador `;` y decimal `,` para abrir
directo en Excel en español:

| Archivo | Contenido |
|---|---|
| `termometro_EPH.csv` | serie trimestral completa: indicadores en % (`pct_`), percentiles (`perc_`), dimensiones, termómetro, media móvil, variación interanual y estado |
| `termometro_EPH_anual.csv` | promedios anuales (sección 11) |
| `termometro_EPH_resumen.md` | lectura del último trimestre (sección 6) |

In [ ]:
os.makedirs(RESULTADOS_DIR, exist_ok=True)
salida = pd.concat([
    ind.add_prefix("pct_"),
    norm.add_prefix("perc_"),
    termo,
], axis=1)
salida.index.name = "trimestre"
opts = dict(sep=";", decimal=",", encoding="utf-8-sig")
archivos = {
    "termometro_EPH.csv": lambda p: salida.to_csv(p, **opts),
    "termometro_EPH_anual.csv": lambda p: anual.to_csv(p, **opts),
    "termometro_EPH_resumen.md": lambda p: open(p, "w", encoding="utf-8").write(resumen + "\n"),
}
for nombre, guardar in archivos.items():
    path = os.path.join(RESULTADOS_DIR, nombre)
    guardar(path)
    print("Guardado:", path)
print("Serie trimestral:", salida.shape, "| anual:", anual.shape)

## Anexo A. Diagnóstico de variables candidatas

Herramienta para evaluar si conviene **sumar un indicador al índice**. No hace falta
correrla en la actualización trimestral: con `CANDIDATAS` vacío no hace nada.

**Cómo usarla:**
1. Calcular el indicador en la sección 3 (`ind_personas` o `ind_hogares`) y agregarlo a
   `COMPLEMENTARIOS` (sección 4); si usa una variable nueva, sumarla a `PERS_COLS`/`HOG_COLS`.
2. Listarlo en `CANDIDATAS` (celda de abajo) con su dimensión, signo (+1 = más alto es
   peor), la variable de origen y el universo donde mirar sus códigos.
3. Correr el notebook y leer la tabla de criterios y el efecto sobre el índice.
4. Si pasa, moverlo a `DIMENSIONES` y repetir el chequeo de sentido (sección 12).

**Criterios (todos deben cumplirse):**
1. **Cobertura:** datos en todos los trimestres desde 2017 (ambos esquemas).
2. **Códigos estables:** la distribución de la variable no cambia bruscamente con el quiebre
   de esquema de 4T2023 (si cambia, el código pudo cambiar de significado). Se evalúa a ojo.
3. **Sentido:** peor (más alto) en la pandemia (2020T2-2021T1) que en el valle 2023T3-T4.
4. **Señal, no ruido:** autocorrelación de orden 1 ≥ 0,3 (la serie tiene persistencia).
5. **Relación con el índice:** correlación de Spearman con el termómetro actual ≥ 0,3.
6. **No redundante:** correlación absoluta < 0,85 con cada indicador que ya está en el índice.

**Evaluaciones realizadas (2026-09, serie 2017T1-2026T1):**

| Candidata | Resultado | Motivo |
|---|---|---|
| Desocupados por despido/cierre (`PP11O==1`) | ✅ **Incorporada a A** (luego ampliada con el código 7, renuncia obligada/pactada = despido encubierto) | Pasa los 6 criterios: códigos estables entre esquemas (12,7% → 14,2% de los desocupados), pandemia 1,44% vs valle 0,56%, autocorrelación 0,76, Spearman con termómetro 0,60, corr. máx. 0,83 (con desocupación). Sin estacionalidad. Con 1+7: pandemia 1,62% vs valle 0,65%, autocorr. 0,74, corr. máx. 0,82. |
| Pluriempleo (`PP03C==2`) | ❌ **Descartada** (queda como complementaria) | Procíclica: cae en pandemia (7,8% vs 11,2% en el valle 2023) y en cada T1 (8,4% vs 10,1%); Spearman con el termómetro −0,05; corr. 0,77 con la tasa de empleo. Sumarla movía el índice hasta 4,8 pts (bajaba la pandemia). |

**Sin evaluar (ideas):** ex-cuentapropistas que cerraron por falta de clientes (`PP11L==1`,
para sumar al indicador de despidos); jóvenes 18-24 que no estudian ni trabajan; asalariados
temporarios (`PP07C==1`, sesgo de composición); desocupación de jefes de hogar (redundante
con la desocupación).

In [ ]:
# Candidatas a evaluar: nombre del indicador (sección 3) -> configuración.
#   dimension: dimensión donde entraría | signo: +1 si más alto es peor
#   variable: variable EPH de origen (para mirar sus códigos por esquema; None = no mirar)
#   universo: "ocupados", "desocupados" o "todos" (filas donde mirar los códigos)
# Ejemplo (ya evaluado y descartado):
# CANDIDATAS = {"Pluriempleo": {"dimension": "C. Estrés de los hogares", "signo": +1,
#                               "variable": "PP03C", "universo": "ocupados"}}
CANDIDATAS = {}

UNIVERSOS = {"ocupados": pers["ESTADO"] == 1, "desocupados": pers["ESTADO"] == 2,
             "todos": pd.Series(True, index=pers.index)}

if not CANDIDATAS:
    print("Sin candidatas para evaluar (CANDIDATAS vacío). Ver instrucciones arriba.")
else:
    # Criterio 2: distribución de códigos antes/después del quiebre de esquema 4T2023
    for c, cfg in CANDIDATAS.items():
        var = cfg.get("variable")
        if var is None:
            continue
        sub = pers[UNIVERSOS[cfg.get("universo", "todos")]]
        dist = (sub.groupby(["esquema", var])["PONDERA"].sum()
                   .groupby(level=0).transform(lambda s: s / s.sum() * 100).unstack(0))
        print(f"\n{c}: distribución de {var} (% ponderado, universo {cfg.get('universo', 'todos')}):")
        print(dist.round(1).to_string())

    filas = []
    for c, cfg in CANDIDATAS.items():
        s, signo = ind[c], cfg["signo"]
        corr_ind = ind[list(SIGNO)].corrwith(s, method="spearman").abs()
        chk = {
            "Primer trimestre con dato": s.first_valid_index(),
            "Trimestres sin dato": int(s.isna().sum()),
            "Media pandemia": s.loc[PANDEMIA].mean(),
            "Media valle 2023": s.loc[VALLE].mean(),
            "Autocorrelación lag 1": s.autocorr(1),
            "Spearman con termómetro": s.corr(termo["Termómetro"], method="spearman"),
            "Máx. corr. con indicador del índice": corr_ind.max(),
            "Indicador más parecido": corr_ind.idxmax(),
        }
        ok = {
            "1. Cobertura": chk["Trimestres sin dato"] == 0 and chk["Primer trimestre con dato"] == ind.index[0],
            "3. Sentido": signo * (chk["Media pandemia"] - chk["Media valle 2023"]) > 0,
            "4. Señal": chk["Autocorrelación lag 1"] >= 0.3,
            "5. Relación": signo * chk["Spearman con termómetro"] >= 0.3,
            "6. No redundante": chk["Máx. corr. con indicador del índice"] < 0.85,
        }
        filas.append({"Candidata": c, **chk, **ok, "Pasa (sin contar criterio 2)": all(ok.values())})
    display(pd.DataFrame(filas).set_index("Candidata").T)

In [ ]:
if CANDIDATAS:
    # Serie de cada candidata, con pandemia y valle 2023 sombreados
    fig, axes = plt.subplots(1, len(CANDIDATAS), figsize=(6.5 * len(CANDIDATAS), 4.5), squeeze=False)
    for ax, c in zip(axes[0], CANDIDATAS):
        for rango, nombre in [(PANDEMIA, "Pandemia"), (VALLE, "Valle 2023")]:
            i0, i1 = ind.index.get_loc(rango[0]), ind.index.get_loc(rango[-1])
            ax.axvspan(i0 - 0.5, i1 + 0.5, color="#e4e3df", lw=0)
            ax.text((i0 + i1) / 2, 1.01, nombre, transform=ax.get_xaxis_transform(),
                    ha="center", va="bottom", color="#52514e", fontsize=8)
        ax.plot(x, ind[c], color=TINTA, lw=2, marker="o", ms=3)
        ax.set_title(c + " (%)", pad=16)
        ax.set_xticks(x[::4], ind.index[::4], rotation=90)
    plt.tight_layout()
    plt.show()

    # Efecto sobre el índice si se agregaran las candidatas
    dims_con = {d: dict(cols) for d, cols in DIMENSIONES.items()}
    for c, cfg in CANDIDATAS.items():
        dims_con[cfg["dimension"]][c] = cfg["signo"]
    _, _, t_con = armar_termometro(dims_con)
    comp = pd.DataFrame({"Actual": termo["Termómetro"], "Con candidatas": t_con})
    comp["Diferencia"] = comp["Con candidatas"] - comp["Actual"]
    print(f"Correlación actual vs. con candidatas: {comp['Actual'].corr(comp['Con candidatas']):.3f} | "
          f"diferencia máx.: {comp['Diferencia'].abs().max():.1f} pts")
    print(f"Máximo: {comp['Actual'].idxmax()} -> {comp['Con candidatas'].idxmax()} | "
          f"mínimo: {comp['Actual'].idxmin()} -> {comp['Con candidatas'].idxmin()}")
    display(comp.loc[["2019T2", *PANDEMIA, *VALLE, "2024T1", ult]].round(1))

## Anexo B. Historial de versiones del índice

| Versión | Cambio | Por qué |
|---|---|---|
| **v1** | 3 dimensiones; A: desocupación, empleo, desocupación > 1 año, desalentados; B: subocupación, buscan otro empleo, sin descuento jubilatorio; C: `V13`, `V14`, `V17`, `V6`, `V7`, niños que aportan | Falló el chequeo de sentido: 2020T2 = 48 (dimensión B = 0,9) por **efecto composición**; desalentados y niños (~0,1%) metían ruido. |
| **v2** | Desalentados y niños pasan a complementarios; B suma la tasa de empleo asalariado registrado sobre población; redondeo a 1 decimal antes de rankear | 2020T2 → 64,7; máximo en 2020T4. |
| **v3** | A suma desocupados por despido/cierre (`PP11O==1`) y renuncia obligada/pactada (`PP11O==7`); pluriempleo evaluado y descartado | Pasó los 6 criterios del Anexo A; adelanta los deterioros (2018T3-T4, 2024T1). |
| v3 (notebook, 2026-09-29) | Sin cambios en el índice. Se suman la lectura automática (6), promedios anuales (11), controles de calidad (12), exportación del anual y del resumen; el diagnóstico pasa a anexo y queda vacío por defecto | Dejar la corrida trimestral limpia y autoexplicada. |

Referencia de resultados v3 (T1-2017 → T1-2026): máximo 2020T4 = 82,1; mínimo 2017T4 = 20,4;
valle 2023T3 = 22,4; 2020T2 = 65,7; 2026T1 = 60,0 (Tibio, +11,0 i.a.). Al sumar trimestres
estos valores pueden moverse levemente (los percentiles se recalculan con toda la historia).